# **【8-2：以少量資料集從頭訓練一個卷積神經網路】**

### **【下載資料，並將其導入Colab中】**

In [11]:
from google.colab import files # 從 google.colab 庫中導入 files 模組，用於 Colab 環境中的檔案操作。
files.upload() # 呼叫 files.upload() 函數，開啟檔案上傳介面，讓使用者可以選擇檔案上傳到 Colab 運行環境中。

Saving kaggle.json to kaggle.json


{'kaggle.json': b'{"username":"tingy0568gmailcom","key":"b96e005f31f08adfe6d9e8faa706ce48"}'}

In [12]:
!mkdir -p ~/.kaggle # 在當前用戶的主目錄下創建一個名為 .kaggle 的隱藏資料夾，如果該資料夾已存在則不會報錯。
!cp kaggle.json ~/.kaggle/ # 將上傳的 kaggle.json 檔案複製到 ~/.kaggle/ 資料夾中。這個檔案包含了 Kaggle API 認證憑證。
!chmod 600 ~/.kaggle/kaggle.json # 將 kaggle.json 檔案的權限設置為 600，表示只有檔案所有者有讀寫權限，其他用戶無任何權限，以保護敏感資訊。

In [13]:
!kaggle competitions download -c dogs-vs-cats # 使用 Kaggle CLI 工具下載 'dogs-vs-cats' 競賽的資料集。

 99% 806M/812M [00:04<00:00, 98.8MB/s]
100% 812M/812M [00:04<00:00, 205MB/s] 


解壓縮資料集

In [14]:
!unzip -qq dogs-vs-cats.zip # 解壓縮 dogs-vs-cats.zip 檔案，'-qq' 參數表示靜默模式，不顯示解壓縮過程中的詳細資訊。
!unzip -qq train.zip # 解壓縮 train.zip 檔案。
!unzip -qq test1.zip # 解壓縮 test1.zip 檔案。

In [15]:
from google.colab import drive # 從 google.colab 庫中導入 drive 模組，用於掛載 Google Drive。
drive.mount('/content/drive') # 呼叫 drive.mount() 函數，將 Google Drive 掛載到 Colab 運行環境的 /content/drive 目錄。這行被註釋掉了，表示當前並未啟用 Google Drive。

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


### **【程式8.6】**

In [16]:
import os,shutil,pathlib # 導入 os 模組用於操作文件系統，shutil 模組用於高級文件操作，pathlib 模組提供物件導向的路徑操作。

original_dir = pathlib.Path('train') # 定義原始訓練資料夾的路徑為 'train'。
new_base_dir = pathlib.Path('cats_vs_dogs_small') # 定義新的小型資料集根目錄的路徑為 'cats_vs_dogs_small'。

def make_subset(subset_name,start_index,end_index): # 定義一個函數，用於創建資料集的子集。
  for category in ('cat','dog'): # 遍歷 'cat' 和 'dog' 兩個類別。
    dir = new_base_dir / subset_name / category # 構建每個類別子集的新資料夾路徑。
    os.makedirs(dir, exist_ok=True) # 創建這些資料夾，如果已存在則不報錯。
    fnames = [f"{category}.{i}.jpg"
      for i in range(start_index,end_index)] # 根據起始和結束索引生成檔案名列表。
    for fname in fnames: # 遍歷生成的檔案名。
      shutil.copyfile(src=original_dir/fname, # 從原始資料夾複製檔案。
                      dst=dir/fname) # 將檔案複製到新創建的子集資料夾中。

make_subset('train',start_index=0,end_index=1000) # 創建訓練子集，包含貓和狗各 1000 張圖片。

make_subset('validation',start_index=1000,end_index=1500) # 創建驗證子集，包含貓和狗各 500 張圖片。

make_subset('test',start_index=1500,end_index=2500) # 創建測試子集，包含貓和狗各 1000 張圖片。

### **【程式8.7】**

In [17]:
from tensorflow import keras # 從 TensorFlow 導入 Keras 模組。
from tensorflow.keras import layers # 從 tensorflow.keras 導入 layers 模組，用於構建神經網路層。

inputs = keras.Input(shape=(180,180,3)) # 定義模型的輸入層，指定輸入圖片的形狀為 180x180 像素，3 個顏色通道 (RGB)。

x = layers.Rescaling(1./255)(inputs) # 將輸入圖片的像素值從 0-255 縮放到 0-1 範圍，進行數據正規化。
x = layers.Conv2D(filters=32,kernel_size=3,activation='relu')(x) # 第一個卷積層：32 個濾波器，卷積核大小 3x3，使用 ReLU 啟動函數。
x = layers.MaxPooling2D(pool_size=2)(x) # 第一個最大池化層：池化窗口大小 2x2，用於下採樣。
x = layers.Conv2D(filters=64,kernel_size=3,activation='relu')(x) # 第二個卷積層：64 個濾波器，卷積核大小 3x3，使用 ReLU 啟動函數。
x = layers.MaxPooling2D(pool_size=2)(x) # 第二個最大池化層。
x = layers.Conv2D(filters=128,kernel_size=3,activation='relu')(x) # 第三個卷積層：128 個濾波器，卷積核大小 3x3，使用 ReLU 啟動函數。
x = layers.MaxPooling2D(pool_size=2)(x) # 第三個最大池化層。
x = layers.Conv2D(filters=256,kernel_size=3,activation='relu')(x) # 第四個卷積層：256 個濾波器，卷積核大小 3x3，使用 ReLU 啟動函數。
x = layers.MaxPooling2D(pool_size=2)(x) # 第四個最大池化層。
x = layers.Conv2D(filters=256,kernel_size=3,activation='relu')(x) # 第五個卷積層：256 個濾波器，卷積核大小 3x3，使用 ReLU 啟動函數。
x = layers.Flatten()(x) # 將卷積層的輸出展平為一維向量，以便連接到全連接層。
outputs = layers.Dense(1,activation='sigmoid')(x) # 輸出層：一個神經元，使用 Sigmoid 啟動函數，用於二元分類（貓或狗）。
model = keras.Model(inputs=inputs,outputs=outputs) # 創建 Keras 模型，指定輸入和輸出。

In [18]:
model.summary() # 顯示模型的摘要資訊，包括各層的名稱、輸出形狀和參數數量。

Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_1 (InputLayer)      │ (None, 180, 180, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ rescaling (Rescaling)           │ (None, 180, 180, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d (Conv2D)                 │ (None, 178, 178, 32)   │           896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d (MaxPooling2D)    │ (None, 89, 89, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_1 (Conv2D)               │ (None, 87, 87, 64)     │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_1 (MaxPooling2D)  │ (None, 43, 43, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_2 (Conv2D)               │ (None, 41, 41, 128)    │        73,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_2 (MaxPooling2D)  │ (None, 20, 20, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_3 (Conv2D)               │ (None, 18, 18, 256)    │       295,168 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_3 (MaxPooling2D)  │ (None, 9, 9, 256)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_4 (Conv2D)               │ (None, 7, 7, 256)      │       590,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 12544)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 1)              │        12,545 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 991,041 (3.78 MB)

 Trainable params: 991,041 (3.78 MB)

 Non-trainable params: 0 (0.00 B)

### **【程式8.8】**

In [19]:
model.compile(loss='binary_crossentropy', # 配置模型的訓練過程：指定損失函數為二元交叉熵，適用於二元分類問題。
              optimizer='rmsprop', # 指定優化器為 RMSprop。
              metrics=['accuracy']) # 指定評估指標為準確度。

### **【程式8.9】**

In [20]:
from tensorflow.keras.utils import image_dataset_from_directory # 從 tensorflow.keras.utils 導入 image_dataset_from_directory 函數，用於從目錄載入圖片資料集。

train_dataset = image_dataset_from_directory( # 載入訓練資料集。
    new_base_dir / 'train', # 指定訓練資料的目錄。
    image_size=(180,180), # 將圖片調整為 180x180 像素。
    batch_size=32 # 設定每個批次的圖片數量為 32。
)
validation_dataset = image_dataset_from_directory( # 載入驗證資料集。
    new_base_dir / 'validation', # 指定驗證資料的目錄。
    image_size=(180,180), # 將圖片調整為 180x180 像素。
    batch_size=32 # 設定每個批次的圖片數量為 32。
)
test_dataset = image_dataset_from_directory( # 載入測試資料集。
    new_base_dir / 'test', # 指定測試資料的目錄。
    image_size=(180,180), # 將圖片調整為 180x180 像素。
    batch_size=32 # 設定每個批次的圖片數量為 32。
)

Found 2000 files belonging to 2 classes.
Found 1000 files belonging to 2 classes.
Found 2000 files belonging to 2 classes.


### **【程式8.10】**

In [21]:
for data_batch,labels_batch in train_dataset: # 遍歷訓練資料集中的一個批次。
  print('data batch shape:',data_batch.shape) # 輸出資料批次的形狀 (batch_size, height, width, channels)。
  print('labels batch shape:',labels_batch.shape) # 輸出標籤批次的形狀 (batch_size,)。
  break # 在獲取第一個批次後立即跳出迴圈，只為了查看形狀。

data batch shape: (32, 180, 180, 3)
labels batch shape: (32,)


### **【程式8.11】**

In [ ]:
from tensorflow import keras # 從 TensorFlow 導入 Keras 模組。

callback = [ # 定義回呼函數列表。
    keras.callbacks.ModelCheckpoint( # ModelCheckpoint 回呼函數用於在訓練過程中儲存模型。
        filepath='convnet_from_scratch.keras', # 指定儲存模型的路徑和檔案名。
        save_best_only=True, # 只儲存驗證損失最小（最佳）的模型。
        monitor='val_loss')] # 監控驗證損失 (validation loss) 來判斷模型好壞。
history = model.fit( # 訓練模型。
    train_dataset, # 指定訓練資料集。
    epochs=30, # 設定訓練週期為 30。
    validation_data=validation_dataset, # 指定驗證資料集，用於在每個週期結束後評估模型性能。
    callbacks=callback # 傳入回呼函數列表。
)

Epoch 1/30
63/63 ━━━━━━━━━━━━━━━━━━━━ 238s 4s/step - accuracy: 0.4961 - loss: 0.6932 - val_accuracy: 0.5080 - val_loss: 0.6926
Epoch 2/30
63/63 ━━━━━━━━━━━━━━━━━━━━ 252s 4s/step - accuracy: 0.5350 - loss: 0.6998 - val_accuracy: 0.5260 - val_loss: 0.6708
Epoch 3/30
63/63 ━━━━━━━━━━━━━━━━━━━━ 257s 4s/step - accuracy: 0.5941 - loss: 0.6792 - val_accuracy: 0.5550 - val_loss: 0.7867
Epoch 4/30
63/63 ━━━━━━━━━━━━━━━━━━━━ 291s 5s/step - accuracy: 0.6682 - loss: 0.6324 - val_accuracy: 0.6530 - val_loss: 0.6215
Epoch 5/30
63/63 ━━━━━━━━━━━━━━━━━━━━ 241s 3s/step - accuracy: 0.6610 - loss: 0.5953 - val_accuracy: 0.6380 - val_loss: 0.6729
Epoch 6/30
63/63 ━━━━━━━━━━━━━━━━━━━━ 219s 3s/step - accuracy: 0.6931 - loss: 0.5899 - val_accuracy: 0.6780 - val_loss: 0.5915
Epoch 7/30
63/63 ━━━━━━━━━━━━━━━━━━━━ 194s 3s/step - accuracy: 0.7131 - loss: 0.5525 - val_accuracy: 0.6650 - val_loss: 0.6280
Epoch 8/30
63/63 ━━━━━━━━━━━━━━━━━━━━ 211s 3s/step - accuracy: 0.7189 - loss: 0.5589 - val_accuracy: 0.7060 - v

### **【程式8.12】**

In [ ]:
import matplotlib.pyplot as plt # 導入 Matplotlib 庫的 pyplot 模組，用於繪圖。
accuracy = history.history['accuracy'] # 從訓練歷史中獲取訓練準確度。
val_accuracy = history.history['val_accuracy'] # 從訓練歷史中獲取驗證準確度。
loss = history.history['loss'] # 從訓練歷史中獲取訓練損失。
val_loss = history.history['val_loss'] # 從訓練歷史中獲取驗證損失。
epochs = range(1,len(accuracy)+1) # 創建一個 epoch 數值的範圍，從 1 到訓練週期數。
plt.plot(epochs,accuracy,'bo',label='Training accuracy') # 繪製訓練準確度，'bo' 表示藍色圓點。
plt.plot(epochs,val_accuracy,'b',label='Validation accuracy') # 繪製驗證準確度，'b' 表示藍色實線。
plt.title('Training and validation accuracy') # 設定圖表標題。
plt.legend() # 顯示圖例。
plt.figure() # 創建一個新的圖表。
plt.plot(epochs,loss,'bo',label='Training loss') # 繪製訓練損失，'bo' 表示藍色圓點。
plt.plot(epochs,val_loss,'b',label='Validation loss') # 繪製驗證損失，'b' 表示藍色實線。
plt.title('Training and validation loss') # 設定圖表標題。
plt.legend() # 顯示圖例。
plt.show() # 顯示所有圖表。

### **【程式8.13】**

In [ ]:
test_model = keras.models.load_model('convnet_from_scratch.keras') # 載入之前訓練的模型。

test_loss,test_acc = test_model.evaluate(test_dataset) # 在測試資料集上評估模型。
print(f'Test accuracy: {test_acc:.3f}') # 印出測試準確度。

### **【程式8.14】**

In [ ]:
from tensorflow import keras # 從 TensorFlow 導入 Keras 模組。
from tensorflow.keras import layers # 從 tensorflow.keras 導入 layers 模組。

data_augmentation = keras.Sequential( # 定義資料擴增層的序列。
    [
        layers.RandomFlip('horizontal'), # 水平翻轉圖片。會隨機將圖片進行水平翻轉，這有助於模型學習到即使物體方向改變，其類別依然不變的特徵。
        layers.RandomRotation(0.1), # 隨機旋轉圖片，角度範圍為 +/- 0.1 弧度。會隨機旋轉圖片，旋轉角度在一個小範圍內（這裡是 +/- 0.1 弧度），這讓模型能夠更好地識別不同角度的物體。
        layers.RandomZoom(0.2), # 隨機縮放圖片，縮放因子範圍為 +/- 0.2。會隨機對圖片進行放大或縮小，縮放比例在 +/- 0.2 的範圍內。這可以幫助模型在不同大小的物體上保持識別能力。
    ]
)

### **【程式8.15】**

In [ ]:
import matplotlib.pyplot as plt # 導入 Matplotlib 庫的 pyplot 模組。

plt.figure(figsize=(10,10)) # 創建一個新的圖表，並設定圖表大小。
for images,_ in train_dataset.take(1): # 從訓練資料集中取出一個批次的圖片。
  for i in range(9): # 遍歷前 9 張圖片。
    augmented_images = data_augmentation(images) # 對圖片執行數據增強。
    ax = plt.subplot(3,3,i+1) # 在 3x3 的網格中創建子圖。
    plt.imshow(augmented_images[0].numpy().astype('uint8')) # 顯示增強後的圖片。
    plt.axis('off') # 關閉座標軸。
plt.show() # 顯示圖片。

### **【程式8.16】**

In [ ]:
from tensorflow import keras # 從 TensorFlow 導入 Keras 模組。
from tensorflow.keras import layers # 從 tensorflow.keras 導入 layers 模組。

input_layer = keras.Input(shape=(180,180,3)) # 定義模型的輸入層，指定輸入圖片的形狀。
x = data_augmentation(input_layer) # 將輸入圖片傳遞給數據增強層。
x = layers.Rescaling(1./255)(x) # 將像素值縮放到 0-1 範圍。
x = layers.Conv2D(filters=32,kernel_size=3,activation='relu')(x) # 第一個卷積層，使用 ReLU 啟動函數。
x = layers.MaxPooling2D(pool_size=2)(x) # 第一個最大池化層。
x = layers.Conv2D(filters=64,kernel_size=3,activation='relu')(x) # 第二個卷積層，使用 ReLU 啟動函數。
x = layers.MaxPooling2D(pool_size=2)(x) # 第二個最大池化層。
x = layers.Conv2D(filters=128,kernel_size=3,activation='relu')(x) # 第三個卷積層，使用 ReLU 啟動函數。
x = layers.MaxPooling2D(pool_size=2)(x) # 第三個最大池化層。
x = layers.Conv2D(filters=256,kernel_size=3,activation='relu')(x) # 第四個卷積層，使用 ReLU 啟動函數。
x = layers.Flatten()(x) # 展平操作。
x = layers.Dropout(0.5)(x) # Dropout 層，以 0.5 的機率隨機關閉神經元，防止過擬合。
outputs = layers.Dense(1,activation='sigmoid')(x) # 輸出層，一個神經元，使用 Sigmoid 啟動函數。
model = keras.Model(inputs=input_layer,outputs=outputs) # 創建 Keras 模型。

model.compile(loss='binary_crossentropy', # 配置模型，損失函數為二元交叉熵。
              optimizer='rmsprop', # 優化器為 RMSprop。
              metrics=['accuracy']) # 評估指標為準確度。

### **【程式8.17】**

In [ ]:
callbacks = [ # 定義回呼函數列表。
    keras.callbacks.ModelCheckpoint( # ModelCheckpoint 回呼函數。
        filepath='convnet_from_scratch_with_augmentation.keras', # 指定儲存模型檔案的路徑。
        save_best_only=True, # 只儲存驗證損失最小的模型。
        monitor='val_loss') # 監控驗證損失。
]

history = model.fit( # 訓練模型。
    train_dataset, # 訓練資料集。
    epochs=100, # 訓練週期為 100。
    validation_data=validation_dataset, # 驗證資料集。
    callbacks=callbacks # 傳入回呼函數列表。
)

### **【程式8.18】**

In [ ]:
import matplotlib.pyplot as plt # 導入 Matplotlib 庫的 pyplot 模組，用於繪圖。
accuracy = history.history['accuracy'] # 從訓練歷史中獲取訓練準確度。
val_accuracy = history.history['val_accuracy'] # 從訓練歷史中獲取驗證準確度。
loss = history.history['loss'] # 從訓練歷史中獲取訓練損失。
val_loss = history.history['val_loss'] # 從訓練歷史中獲取驗證損失。
epochs = range(1,len(accuracy)+1) # 創建一個 epoch 數值的範圍，從 1 到訓練週期數。
plt.plot(epochs,accuracy,'bo',label='Training accuracy') # 繪製訓練準確度，'bo' 表示藍色圓點。
plt.plot(epochs,val_accuracy,'b',label='Validation accuracy') # 繪製驗證準確度，'b' 表示藍色實線。
plt.title('Training and validation accuracy') # 設定圖表標題。
plt.legend() # 顯示圖例。
plt.figure() # 創建一個新的圖表。
plt.plot(epochs,loss,'bo',label='Training loss') # 繪製訓練損失，'bo' 表示藍色圓點。
plt.plot(epochs,val_loss,'b',label='Validation loss') # 繪製驗證損失，'b' 表示藍色實線。
plt.title('Training and validation loss') # 設定圖表標題。
plt.legend() # 顯示圖例。
plt.show() # 顯示所有圖表。

### **【程式8.19】**

In [ ]:
from tensorflow import keras # 從 TensorFlow 導入 Keras 模組。
import os # 導入 os 模組。

model_filepath = 'convnet_from_scratch_with_augmentation.keras' # 定義模型檔案路徑。

if os.path.exists(model_filepath): # 檢查模型檔案是否存在。
    test_model = keras.models.load_model(model_filepath) # 載入最佳模型。
    test_loss,test_acc = test_model.evaluate(test_dataset) # 在測試資料集上評估模型。
    print(f'Test accuracy: {test_acc:.3f}') # 印出測試準確度。
else: # 如果檔案不存在。
    print(f"Error: Model file '{model_filepath}' not found.") # 印出錯誤訊息。
    print("Please ensure the training cell (cell 5k71mQoYfZ7e) was executed successfully to save the model.") # 提醒用戶。

# **【8-3：使用預先訓練好的模型】**

In [ ]:
import keras
# 這一行導入了 Keras 深度學習函式庫，讓我們可以使用其中的功能，例如預訓練模型。

conv_base = keras.applications.vgg16.VGG16(
# 這一行初始化了一個 VGG16 卷積神經網路模型。VGG16 是一個在 ImageNet 資料集上預訓練過的經典模型。
weights='imagenet',
# 這個參數指定了模型要加載的權重。'imagenet' 表示使用在 ImageNet 資料集上訓練好的權重，這是一個大型的圖像分類資料集。
include_top=False,
# 這個參數設定為 False 意味著我們不包括模型的頂部（即分類層）。當我們將預訓練模型作為特徵提取器用於新的分類任務時，通常會這麼做，因為我們需要添加自定義的分類器。
input_shape=(180, 180, 3))
# 這個參數定義了模型輸入圖像的形狀。這裡設定為 (180, 180, 3) 表示輸入圖像的高度為 180 像素，寬度為 180 像素，3 表示圖像有三個顏色通道（紅、綠、藍）。

In [ ]:
conv_base.summary()

In [ ]:
import numpy as np # 導入 numpy 以創建佔位符數據

def get_features_and_labels(dataset):
  all_features = []
  all_labels = []
  for images,labels in dataset:
    preprocessed_images = keras.applications.vgg16.preprocess_input(images)
    # 這一行使用 VGG16
    features = conv_base.predict(preprocessed_images)
    all_features.append(features)
    all_labels.append(labels)
  return np.concatenate(all_features), np.concatenate(all_labels)

train_features, train_labels = get_features_and_labels(train_dataset)
val_features, val_labels = get_features_and_labels(validation_dataset)
test_features, test_labels = get_features_and_labels(test_dataset)

In [ ]:
train_features.shape

In [ ]:
input = keras.Input(shape=(5, 5, 512))
x = layers.Flatten()(input)
x = layers.Dense(256)(x)
x = layers.Dropout(0.5)(x)
output = layers.Dense(1, activation='sigmoid')(x)
model = keras.Model(input, output)
model.compile(optimizer='rmsprop', loss='binary_crossentropy', metrics=['accuracy'])

callbacks = [
    keras.callbacks.ModelCheckpoint(
        filepath='conv_base_from_scratch.keras',
        save_best_only=True,
        monitor='val_loss')
]
history = model.fit(
    train_features, train_labels,
    epochs=20,
    validation_data=(val_features, val_labels),
    callbacks=callbacks
    )

In [ ]:
import matplotlib.pyplot as plt
acc = history.history['accuracy']
val_acc = history.history['val_accuracy']
loss = history.history['loss']
val_loss = history.history['val_loss']

epochs = range(1, len(acc) + 1)

plt.plot(epochs, acc, 'bo', label='Training accuracy')
plt.plot(epochs, val_acc, 'b', label='Validation accuracy')
plt.title('Training and validation accuracy')
plt.legend()
plt.figure()
plt.plot(epochs, loss, 'bo', label='Training loss')
plt.plot(epochs, val_loss, 'b', label='Validation loss')
plt.title('Training and validation loss')
plt.legend()
plt.show()

In [ ]:
con_base = keras.applications.vgg16.VGG16(
    weights='imagenet',
    include_top=False)
con_base.trainable = False

In [ ]:
conv_base.trainble = True
print("This is the number of trainable weights"
    "before freezing the conv base:",len(conv_base.trainable_weights))
conv_base.trainable = False
print("This is the number of trainable weights"
    "after freezing the conv base:",len(conv_base.trainable_weights))

In [ ]:
data_augmentation = keras.Sequential(
    [
        layers.RandomFlip("horizontal"),
        layers.RandomRotation(0.1),
        layers.RandomZoom(0.2),
    ]
)

inputs = keras.Input(shape=(180, 180, 3))
x = data_augmentation(inputs)
x = keras.applications.vgg16.preprocess_input(x)
x = conv_base(x)
x = layers.Flatten()(x)
x = layers.Dense(256)(x)
x = layers.Dropout(0.5)(x)
outputs = layers.Dense(1, activation="sigmoid")(x)
model = keras.Model(inputs, outputs)
model.compile(loss="binary_crossentropy",
              optimizer="rmsprop",
              metrics=["accuracy"])

In [ ]:
callbacks = [
    keras.callbacks.ModelCheckpoint(
        filepath="feature_extraction_with_data_augmentation.keras",
        save_best_only=True,
        monitor="val_loss")
]
history = model.fit(
    train_dataset,
    epochs=50,
    validation_data=validation_dataset,
    callbacks=callbacks)

In [ ]:
test_model = keras.models.load_model(
    "feature_extraction_with_data_augmentation.keras")
test_loss, test_acc = test_model.evaluate(test_dataset)
print(f"Test accuracy: {test_acc:.3f}")

In [ ]:
conv_base.summary()

In [ ]:
conv_base.trainable = True
for layer in conv_base.layers[:-4]:
    layer.trainable = False

In [ ]:
model.compile(loss="binary_crossentropy",
              optimizer=keras.optimizers.RMSprop(learning_rate=1e-5),
              metrics=["accuracy"])
callbacks = [
    keras.callbacks.ModelCheckpoint(
        filepath="fine_tuning.keras",
        save_best_only=True,
        monitor="val_loss")
]
history = model.fit(
    train_dataset,
    epochs=30,
    validation_data=validation_dataset,
    callbacks=callbacks
    )

In [ ]:
model = keras.models.load_model("fine_tuning.keras")
test_loss, test_acc = model.evaluate(test_dataset)
print(f"Test accuracy: {test_acc:.3f}")